### Part 1 - Test prompts

In [7]:
import json
from tqdm import tqdm
from openai import OpenAI
import random

In [15]:
# Create one prompt to test

# Path for input data and to store prompts
path_medical_ai_tasks = "../resources/medical_ai_tasks.json"
path_medical_topics = "../resources/medical_topics.json"
path_moove_examples = "../resources/generated_doctors_questions.jsonl"

output_path = "../results/batched_prompts_test.jsonl"

# Load data
with open(path_medical_ai_tasks, "r") as f:
    medical_ai_tasks = json.load(f)
ai_task = random.sample(medical_ai_tasks, 1)[0]
with open(path_medical_topics, "r") as f:
    medical_topics = json.load(f)
med_topic = random.sample(medical_topics, 1)[0]
with open(path_moove_examples, "r") as f:
    moove_examples = [json.loads(line) for line in f]
# Define number of few-shoot examples to be added
num_examples = 2

# Helper function to get uniformly at random num_entries moove examples
def get_random_entries(moove_examples, num_examples=2):
    """
    Returns a specified number of random entries from moove_examples without replacement.

    param moove_examples: List of JSON objects loaded from a .jsonl file.
    param num_entries: Number of random entries to return.
    return: List of randomly selected entries.
    """
    if len(moove_examples) < num_examples:
        raise ValueError(f"Not enough entries in moove_examples to select {num_examples} unique entries.")
    
    return random.sample(moove_examples, num_examples)



test_content = "You are an assistant responsible for creating prompts that healthcare workers would ask a medical AI chatbot."
def get_prompt(task, description, additional_instruction, topic, num_examples, example_1, example_2):
    prompt = f'''Generate a prompt that a physician might ask an AI chatbot when tasked with {task} in the context of the medical topic "{topic}".
{task} is described as: {description}
To create a realistic prompt, follow these additional instructions: {additional_instruction}
Only include the generated prompt, adding extra details only if explicitly instructed. Focus solely on generating a realistic prompt a physician might ask a medical AI chatbot.
Below there are {num_examples} examples of real prompts that physicians have previously asked to the medical AI chatbot:
{example_1}
{example_2}
The examples provided are likely not directly related to "{task}" in the context of "{topic}", but they are representative of the format and style physicians use. The prompt you generate should align with the format and style of the {num_examples} examples provided above.'''
    return prompt


example_1, example_2 = get_random_entries(moove_examples, num_examples)
test_prompt = get_prompt(ai_task["task"], ai_task["description"], ai_task["additional_instruction"],
                         med_topic["topic"], num_examples, example_1["question"], example_2["question"])
print(test_prompt)


Generate a prompt that a physician might ask an AI chatbot when tasked with Treatment Recommendations in the context of the medical topic "Medical Education".
Treatment Recommendations is described as: Suggesting evidence-based treatment options for various conditions.
To create a realistic prompt, follow these additional instructions: Generate a realistic medical history and presenting issues for a patient, and base your question on the generated case.
Only include the generated prompt, adding extra details only if explicitly instructed. Focus solely on generating a realistic prompt a physician might ask a medical AI chatbot.
Below there are 2 examples of real prompts that physicians have previously asked to the medical AI chatbot:
As a general practitioner in a rural Australian clinic, I've been following a 75-year-old patient with chronic kidney disease (CKD) stage IIIb, hypertension, and hyperlipidemia. At the latest visit, the patient's eGFR has declined to 30 mL/min/1.73m², promp

In [16]:
# Test version of interacting with the openai API
# Generates only 1 output and prints it

path_to_api_key: str = "../API_KEY.txt"
my_api_key = open(path_to_api_key, 'r').read()
client = OpenAI(api_key=my_api_key)

# Specify which model should be used to answer prompts
gpt_model: str = "gpt-4o"
content = test_content
prompt = test_prompt


print("Send prompt to GPT:")
print("#########")
print(prompt)
print("########")

completion = client.chat.completions.create(
    model= gpt_model,
    messages=[
        {"role": "system", "content": content},
        {
            "role": "user",
            "content": f"{prompt}"
        }
    ]
)
print("Receiving responses from GPT...")
print(f"Print response from {gpt_model:}")
print("************************************")
print(completion.choices[0].message.content)
print("************************************")

Send prompt to GPT:
#########
Generate a prompt that a physician might ask an AI chatbot when tasked with Treatment Recommendations in the context of the medical topic "Medical Education".
Treatment Recommendations is described as: Suggesting evidence-based treatment options for various conditions.
To create a realistic prompt, follow these additional instructions: Generate a realistic medical history and presenting issues for a patient, and base your question on the generated case.
Only include the generated prompt, adding extra details only if explicitly instructed. Focus solely on generating a realistic prompt a physician might ask a medical AI chatbot.
Below there are 2 examples of real prompts that physicians have previously asked to the medical AI chatbot:
As a general practitioner in a rural Australian clinic, I've been following a 75-year-old patient with chronic kidney disease (CKD) stage IIIb, hypertension, and hyperlipidemia. At the latest visit, the patient's eGFR has decli